# Dataport Cleaning Notebook

This notebook cleans raw Dataport or Pecan Street records and reshapes them into a wide table for downstream experiments.


In [ ]:
import pandas as pd
import numpy as np

def clean_pecan_street_data(raw_csv_path, output_csv_path):
    print("1. 开始读取原始 Dataport 数据...")
    # 读取数据（这里自动推断时间格式，处理带时区的字符串）
    df = pd.read_csv(raw_csv_path)
    
    print("2. 提取核心物理量并计算总负荷...")
    # 将时间列转为标准 datetime 并去除时区后缀以便处理
    df['timestamp'] = pd.to_datetime(df['local_15min'], utc=True).dt.tz_convert(None)
    
    # 合并可能存在的多路光伏 (solar, solar2)
    solar_cols = [c for c in df.columns if 'solar' in c.lower()]
    df['pv'] = df[solar_cols].fillna(0).sum(axis=1)
    
    # 物理重构总负荷: Load = Grid + PV
    # Pecan Street 中 grid 为正表示买电，solar 为正表示发电
    if 'grid' in df.columns:
        df['load'] = df['grid'] + df['pv']
    else:
        raise ValueError("数据缺失核心的 'grid' 列，无法重构负荷！")
        
    # 只保留我们关心的三列
    df = df[['timestamp', 'dataid', 'load', 'pv']]
    
    print("3. 透视为宽格式矩阵 (Wide Format)...")
    # 把 dataid 变到列上
    df_pivot = df.pivot_table(index='timestamp', columns='dataid', values=['load', 'pv'])
    
    # 展平多层列索引，重命名为 load_661, pv_661 等格式
    df_pivot.columns = [f"{col[0]}_{col[1]}" for col in df_pivot.columns]
    
    print("4. 强制时间对齐与断层填补 (解决时间跳跃问题)...")
    # 获取数据的绝对起点和终点
    start_time = df_pivot.index.min()
    end_time = df_pivot.index.max()
    
    # 生成一个严格没有间断的 15 分钟连续时间轴
    full_time_index = pd.date_range(start=start_time, end=end_time, freq='15min')
    
    # 用这个完美的连续时间轴重新索引我们的数据（缺失的时间点会被强制暴露为 NaN）
    df_pivot = df_pivot.reindex(full_time_index)
    
    # 对暴露出来的 NaN 进行物理填补
    for col in df_pivot.columns:
        if 'pv' in col:
            # 光伏缺失：大概率是晚上或者通信故障。安全起见，补 0
            df_pivot[col] = df_pivot[col].fillna(0.0)
        elif 'load' in col:
            # 负荷缺失：短时间缺失用线性插值，长时间缺失用均值兜底
            # interpolate(limit=8) 表示只插值最多两小时的空缺
            df_pivot[col] = df_pivot[col].interpolate(method='linear', limit=8)
            # 对于像你数据里那种断层 20 小时的，用该用户的全局平均负荷来兜底
            df_pivot[col] = df_pivot[col].fillna(df_pivot[col].mean())

    print("5. 生成虚拟全局电价信号...")
    # 由于原始数据没带电价，我们生成一段与环境 96 步契合的模拟峰谷电价
    time_array = np.arange(len(df_pivot))
    # 正弦波动电价：模拟中午低电价，傍晚高电价
    fake_price = 15.0 + 8.0 * np.sin(2 * np.pi * time_array / 96 - np.pi/2) 
    df_pivot.insert(0, 'price', fake_price)

    # 导出最终给 RL 环境使用的数据
    df_pivot.to_csv(output_csv_path, index_label='timestamp')
    print(f"清洗完成！完美对齐的数据已保存至: {output_csv_path}")
    print(f"最终数据形状: {df_pivot.shape}")
    
    return df_pivot

# === 调用执行 ===
df_clean = clean_pecan_street_data('./data/15minute_data_austin.csv', './data/train_data.csv')
df_clean.head(10)